# Search strategy comparison

The [hyperparameter search](hyperparameter-search.ipynb) chapter introduced grid,
random, and TPE search. Here we put them head-to-head on the **same objective**
with the **same budget** (number of evaluations), measuring both **quality**
(best error found) and **cost** (wall-clock time) — because a search strategy is
only worth its complexity if it earns back the time it spends.

In [ ]:
:dep tpe = { version = "0.3" }
:dep rand = { version = "0.10" }
use std::time::Instant;

// Synthetic validation error (lower is better), minimum near x = 2.
fn objective(x: f64) -> f64 { (x - 2.0).powi(2) * 0.1 + 0.05 + 0.03 * (x * 4.0).sin() }

let budget = 40usize;

// Each search records best-error-so-far after every evaluation (an 'any-time' curve).
let t = Instant::now();
let grid_curve: Vec<f64> = {
    let mut best = f64::INFINITY;
    (0..budget).map(|i| { best = best.min(objective(5.0 * i as f64 / (budget - 1) as f64)); best }).collect()
};
let grid_time = t.elapsed();

let t = Instant::now();
let random_curve: Vec<f64> = {
    use rand::{RngExt, SeedableRng};
    let mut rng = rand::rngs::StdRng::seed_from_u64(0);
    let mut best = f64::INFINITY;
    (0..budget).map(|_| { let x: f64 = rng.random_range(0.0..5.0); best = best.min(objective(x)); best }).collect()
};
let random_time = t.elapsed();

let t = Instant::now();
let tpe_curve: Vec<f64> = {
    use tpe::{TpeOptimizer, parzen_estimator, range};
    use rand::SeedableRng;
    let mut optim = TpeOptimizer::new(parzen_estimator(), range::Range::new(0.0, 5.0).unwrap());
    let mut rng = rand::rngs::StdRng::seed_from_u64(0);
    let mut best = f64::INFINITY;
    (0..budget).map(|_| { let x = optim.ask(&mut rng).unwrap(); let e = objective(x); optim.tell(x, e).unwrap(); best = best.min(e); best }).collect()
};
let tpe_time = t.elapsed();

println!("{:<8}  {:>9}  {:>12}", "strategy", "best err", "time");
println!("{:<8}  {:>9.4}  {:>12?}", "grid",   grid_curve[budget - 1],   grid_time);
println!("{:<8}  {:>9.4}  {:>12?}", "random", random_curve[budget - 1], random_time);
println!("{:<8}  {:>9.4}  {:>12?}", "tpe",    tpe_curve[budget - 1],    tpe_time);

## Any-time performance

The chart below plots **best error found so far** against **number of
evaluations** for each strategy (grid = red, random = blue, TPE = green). A curve
that drops faster reaches a good answer with fewer evaluations — the property
that matters when each evaluation is an expensive model fit:

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

evcxr_figure((520, 360), |root| {
    root.fill(&WHITE)?;
    let max_y = grid_curve[0].max(random_curve[0]).max(tpe_curve[0]) * 1.05;
    let mut chart = ChartBuilder::on(&root)
        .caption("Best error vs evaluations (grid=red, random=blue, tpe=green)", ("sans-serif", 15))
        .margin(10).x_label_area_size(35).y_label_area_size(45)
        .build_cartesian_2d(0f64..budget as f64, 0f64..max_y)?;
    chart.configure_mesh().x_desc("evaluations").y_desc("best error").draw()?;
    chart.draw_series(LineSeries::new((0..budget).map(|i| (i as f64, grid_curve[i])), &RED))?;
    chart.draw_series(LineSeries::new((0..budget).map(|i| (i as f64, random_curve[i])), &BLUE))?;
    chart.draw_series(LineSeries::new((0..budget).map(|i| (i as f64, tpe_curve[i])), &GREEN))?;
    Ok(())
})

## When to use which

Cost and quality together give practical guidance:

- **Grid search** — small, low-dimensional spaces where being exhaustive is
  affordable and reproducibility matters. Cheap per evaluation, but the number of
  points explodes with dimensions.
- **Random search** — a strong, simple default for larger / higher-dimensional
  spaces; near-zero overhead and usually better any-time performance than grid.
- **TPE (Bayesian)** — when each evaluation is *expensive* (e.g. a full
  cross-validated model fit), so it's worth spending extra bookkeeping time to
  need fewer evaluations. Note in the table that TPE's per-evaluation overhead
  makes it the slowest here on a *cheap* objective — that overhead only pays off
  when the objective itself is costly.

```{warning}
**Ecosystem maturity.** Rust's hyperparameter-optimization tooling is still much
thinner than Python's Optuna / Hyperopt / Ray Tune. `tpe` is a focused,
single-algorithm crate, not a full framework — re-verify its state before
relying on it. For parallelizing grid/random search, see the
[Multithreading chapter](../04b-multithreading/parallel-ml.ipynb).
```